# Fase 6 — Da planilha para a fila do banco (CRISP-DM)

**Em uma frase:** o [05](05_evaluation.ipynb) escolheu o aluno e a **linha**. Aqui pensamos como essa decisão chega numa ficha **nova** — sem montar um site, sem inventar infraestrutura. O foco é o *ciclo de vida*, não o Docker.

**Como estudar:** um bloco por vez. Leia → rode → leia o “como ler”. Kernel: `.venv`. Este caderno **não** retreina.

---

## O que este caderno faz — e o que ele *não* faz

| Faz | Não faz (de propósito) |
|---|---|
| Juntar o **pacote** que viaja junto (aluno + régua do 03 + linha do 05) | API REST, fila Kafka, nuvem |
| Decidir **uma ficha nova** com a mesma receita do 03 | Recalcular mediana/P99 (“o mundo mudou”) |
| Explicar um “não” com os pesos da logística | SHAP / LIME (opcional no 04) |
| Combinar o que vigiar daqui a 3 meses | Dashboard pronto |
| Fechar o CRISP-DM: o que aprendemos, o que faríamos diferente | Dizer que o projeto “acabou para sempre” |

Se um dia você empacotar isso numa API, a função da seção 3 *é* o miolo. O resto é encanamento.

---

## A viagem inteira, em uma mesa

| Caderno | Entregou | O 06 carrega |
|---|---|---|
| [01](01_business_understanding.ipynb) | FN ~20× FP. Alvo: fila > 0,80, KS > 30%, apontar ≥ 75%. | A linha e o custo. O texto ao cliente. |
| [02](02_data_understanding.ipynb) | Cartão e atraso 90+ mandam; renda em branco **não** é mais risco; 96/98 são código. | Se a ficha nova vier com 98, **não** é “atrasou 98 vezes”. |
| [03](03_data_preparation.ipynb) | Régua só do treino: 5400, tetos, bandeirinhas. | `impute_params.json`. **A mesma régua** em toda ficha nova. |
| [04](04_modeling.ipynb) | Motor = boosting. Explicação = logística com peso. | `hist_gb.joblib`, `logreg_balanced.joblib`. |
| [05](05_evaluation.ipynb) | Linha **0,43** (treino). Checklist do 01: três sim. | `politica_fase5.json`. |

**A ideia que destrava produção:** o aluno sozinho **não** decide. Sem a régua do 03, uma renda em branco vira lixo ou vira 5400 *sem* bandeirinha. Sem a linha do 05, alguém usa 50% de novo.


---
# 1. O pacote que viaja junto


## 1.1 O que *tem* que ir para o servidor (mesmo que o servidor seja esta pasta)

Pense numa caixa. Se faltar uma peça, a decisão muda e ninguém percebe.

| Peça | Arquivo | Por que não pode faltar |
|---|---|---|
| Motor | `models/hist_gb.joblib` | Quem dá a **nota** |
| Conta explicável | `models/logreg_balanced.joblib` | Quem ajuda a dizer o “porquê” |
| Política | `models/politica_fase5.json` | A **linha** 0,43 + custo 20/1 |
| Régua do 03 | `data/processed/impute_params.json` | Mediana 5400, tetos, moda 0 |
| Lista das 13 colunas | dentro do `metrics_fase4.json` | Ordem e nomes iguais aos do `fit` |
| Semente da história | este repositório + os cadernos | Daqui a 6 meses você (ou outra pessoa) reproduz |

**Versionar** = gravar *juntos* aluno + régua + linha + data. Trocar só o `.joblib` e manter a mediana velha é um bug silencioso.

Não versionamos o `X_train.csv` no servidor de decisão. O aluno já “comeu” o treino. O que entra é **uma ficha de cada vez** (ou um lote do dia).


## 1.2 Abrir a caixa e conferir o lacre


In [1]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd


def project_root() -> Path:
    """Sobe pastas até achar data/processed/X_train.csv."""
    here = Path.cwd().resolve()
    for pasta in [here, *here.parents]:
        if (pasta / "data" / "processed" / "X_train.csv").exists():
            return pasta
    raise FileNotFoundError("Rode o 03 e o 04/05 antes. Falta data/processed/.")


ROOT = project_root()
PROCESSED = ROOT / "data" / "processed"
MODELS = ROOT / "models"

regua = json.loads((PROCESSED / "impute_params.json").read_text())
politica = json.loads((MODELS / "politica_fase5.json").read_text())
colunas = json.loads((MODELS / "metrics_fase4.json").read_text())["features"]

motor = joblib.load(MODELS / "hist_gb.joblib")
explicavel = joblib.load(MODELS / "logreg_balanced.joblib")

print("régua do 03:", regua)
print()
print("política do 05:")
print("  motor     :", politica["motor"])
print("  linha     :", politica["corte"], "(nascida no", politica["corte_foi_escolhido_em"] + ")")
print("  custo FN/FP:", politica["custo_fn"], "/", politica["custo_fp"])
print("  checklist :", politica["checklist_01"])
print()
print("13 colunas, nesta ordem:")
for i, c in enumerate(colunas, 1):
    print(f"  {i:2d}. {c}")

assert politica["motor"] == "hist_gb"
assert abs(politica["corte"] - 0.43) < 1e-9
assert colunas[0] == "RevolvingUtilizationOfUnsecuredLines"
print()
print("lacre ok — é a receita desta turma.")


régua do 03: {'mediana_renda': 5400.0, 'moda_dep': 0.0, 'p99_util': 1.0959067669800004, 'p99_debt_ok': 655.0, 'p99_debt_na': 8137.649999999994, 'p99_renda': 23018.30000000006}

política do 05:
  motor     : hist_gb
  linha     : 0.43 (nascida no treino)
  custo FN/FP: 20 / 1
  checklist : {'auc_acima_0_80': True, 'ks_acima_0_30': True, 'recall_acima_0_75': True}

13 colunas, nesta ordem:
   1. RevolvingUtilizationOfUnsecuredLines
   2. age
   3. NumberOfTime30-59DaysPastDueNotWorse
   4. DebtRatio
   5. MonthlyIncome
   6. NumberOfOpenCreditLinesAndLoans
   7. NumberOfTimes90DaysLate
   8. NumberRealEstateLoansOrLines
   9. NumberOfTime60-89DaysPastDueNotWorse
  10. NumberOfDependents
  11. income_was_missing
  12. dependents_was_missing
  13. util_gt_10

lacre ok — é a receita desta turma.


### Como ler o lacre

Se a linha não for 0,43 ou o motor não for `hist_gb`, alguém rerodou o 04/05 com outra semente. **Pare.** Decidir com a caixa errada é pior do que não decidir.

A régua tem que ser a do treino: renda do meio **5400**, cartão no teto **~1,10**, dívida **655** (com renda) / **~8138** (sem renda). Esses números **não** se recalculam em cada ficha nova — senão cada cliente teria a sua própria mediana. É o mesmo pecado do leakage, agora em produção.


---
# 2. Dois jeitos de entrar na fila do banco


O aluno é o mesmo. Muda *quando* a nota é pedida.

| | Lote (batch) | Uma a uma (API) |
|---|---|---|
| Quando | De madrugada, nas propostas do dia | No clique “enviar proposta” |
| Volume | Milhares de fichas de uma vez | Uma ficha, resposta em fração de segundo |
| Serve para | Cartão já emitido, revisão de limite, cobrança | Crédito na hora (fintech, checkout) |
| Se o aluno cair | Reprocessa o lote de manhã | A loja para — precisa de plano B |

**Neste projeto o lote basta.** Give Me Some Credit é “risco em 2 anos”, não “aprovar PIX em 200 ms”. Uma função Python que lê um CSV do dia e devolve “libera / recusa” já é implantação de verdade.

O 01 ainda manda: recusar um caloteiro vale ~20 recusas de gente boa. Em lote isso vira **política da carteira**. Em API isso vira **texto na tela**. Os números são os mesmos; o tom da conversa muda.


---
# 3. A receita numa ficha **nova** (o miolo da implantação)


## 3.1 Por que não podemos jogar a ficha crua no aluno

O boosting foi ensinado com **13** colunas já limpas. A ficha que chega do cadastro tem **10** e ainda pode ter:

- renda em branco
- 96/98 nas colunas de atraso (código, não atraso)
- cartão em 5.000 (lixo)
- dívida gigante porque a renda estava vazia (os dois mundos do 03)

Se você der `predict` nisso, o Python até roda — e a decisão é **lixo educado**.

A ordem é a do 03, congelada:

1. 96/98 → “sem atraso observado” (vira 0)
2. **Marcar** bandeirinhas *antes* de preencher
3. Preencher renda com **5400** e dependentes com **0** (régua do treino)
4. Teto no cartão, na dívida (dois tetos) e na renda
5. Só então pedir a nota
6. Recusar se nota **≥ 0,43**


## 3.2 As funções — leia os comentários, são a aula


In [2]:
COLUNAS_CRUAS = [
    "RevolvingUtilizationOfUnsecuredLines",
    "age",
    "NumberOfTime30-59DaysPastDueNotWorse",
    "DebtRatio",
    "MonthlyIncome",
    "NumberOfOpenCreditLinesAndLoans",
    "NumberOfTimes90DaysLate",
    "NumberRealEstateLoansOrLines",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfDependents",
]
ATRASOS = [
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate",
]


def preparar_ficha(ficha: dict, regua: dict, colunas: list[str]) -> pd.DataFrame:
    """Aplica a receita do 03 numa ficha crua. A régua NÃO é recalculada."""
    row = {c: ficha.get(c, np.nan) for c in COLUNAS_CRUAS}
    s = pd.Series(row, dtype="float64")

    # 96/98 = código de sistema (02/03), não "atrasou 98 vezes".
    for c in ATRASOS:
        if pd.notna(s[c]) and s[c] >= 96:
            s[c] = np.nan

    # Bandeirinhas ANTES de preencher — senão "branco" e "ganha 5400" viram a mesma pessoa.
    income_na = int(pd.isna(s["MonthlyIncome"]))
    dep_na = int(pd.isna(s["NumberOfDependents"]))
    util_lixo = int(s["RevolvingUtilizationOfUnsecuredLines"] > 10) if pd.notna(
        s["RevolvingUtilizationOfUnsecuredLines"]
    ) else 0

    s["MonthlyIncome"] = s["MonthlyIncome"] if pd.notna(s["MonthlyIncome"]) else regua["mediana_renda"]
    s["NumberOfDependents"] = s["NumberOfDependents"] if pd.notna(s["NumberOfDependents"]) else regua["moda_dep"]
    for c in ATRASOS:
        if pd.isna(s[c]):
            s[c] = 0.0

    # Tetos do TREINO (P99). Cliente novo não empurra o teto.
    s["RevolvingUtilizationOfUnsecuredLines"] = min(
        s["RevolvingUtilizationOfUnsecuredLines"], regua["p99_util"]
    )
    teto_divida = regua["p99_debt_na"] if income_na else regua["p99_debt_ok"]
    s["DebtRatio"] = min(s["DebtRatio"], teto_divida)
    s["MonthlyIncome"] = min(s["MonthlyIncome"], regua["p99_renda"])

    s["income_was_missing"] = income_na
    s["dependents_was_missing"] = dep_na
    s["util_gt_10"] = util_lixo

    # Uma linha, 13 colunas, na ORDEM do fit. Trocar ordem muda a nota.
    return pd.DataFrame([s.reindex(colunas).tolist()], columns=colunas)


def decidir(ficha: dict) -> dict:
    """Miolo de produção: limpa → nota do boosting → linha do 05."""
    X = preparar_ficha(ficha, regua, colunas)
    nota = float(motor.predict_proba(X)[0, 1])
    corte = float(politica["corte"])
    recusa = nota >= corte
    return {
        "nota_de_risco": round(nota, 4),
        "corte": corte,
        "decisao": "RECUSAR" if recusa else "LIBERAR",
        "ficha_limpa": X.iloc[0].to_dict(),
    }


print("funções prontas. Uma ficha crua entra, uma decisão sai.")


funções prontas. Uma ficha crua entra, uma decisão sai.


### Como ler o código (sem ser aula de Python)

Três regras de ouro, iguais às do 03:

1. **Bandeirinha antes do preenchimento.** Sem isso, 20% das fichas novas ficam iguais a “ganha 5.400”.
2. **Teto do treino, não o teto de hoje.** Se um cliente chegar com cartão 50.000, cortamos em ~1,10 e marcamos `util_gt_10`.
3. **A ordem das 13 colunas é contrato.** O boosting não lê o *nome* com carinho; lê a posição que viu no `fit`.


## 3.3 Três fichas inventadas (e uma real da prova)


Três pessoas de mentira, para ver a política se mexer. Os números são caricaturalmente o gabarito do 02:

- **Ana** — 62 anos, cartão baixo, zero atraso, renda informada.
- **Bruno** — 28 anos, cartão estourado, já atrasou 90+ dias.
- **Carla** — renda em branco, dívida gigante (o “segundo mundo” do 03).

Depois, uma linha **de verdade** da prova: em produção você *não* saberia o calote por 2 anos. Aqui olhamos o gabarito só para aprender.


In [3]:
ana = {
    "RevolvingUtilizationOfUnsecuredLines": 0.08,
    "age": 62,
    "NumberOfTime30-59DaysPastDueNotWorse": 0,
    "DebtRatio": 0.22,
    "MonthlyIncome": 8200,
    "NumberOfOpenCreditLinesAndLoans": 6,
    "NumberOfTimes90DaysLate": 0,
    "NumberRealEstateLoansOrLines": 1,
    "NumberOfTime60-89DaysPastDueNotWorse": 0,
    "NumberOfDependents": 0,
}
bruno = {
    "RevolvingUtilizationOfUnsecuredLines": 1.15,
    "age": 28,
    "NumberOfTime30-59DaysPastDueNotWorse": 2,
    "DebtRatio": 0.55,
    "MonthlyIncome": 2800,
    "NumberOfOpenCreditLinesAndLoans": 8,
    "NumberOfTimes90DaysLate": 1,
    "NumberRealEstateLoansOrLines": 0,
    "NumberOfTime60-89DaysPastDueNotWorse": 1,
    "NumberOfDependents": 1,
}
carla = {
    "RevolvingUtilizationOfUnsecuredLines": 0.30,
    "age": 45,
    "NumberOfTime30-59DaysPastDueNotWorse": 0,
    "DebtRatio": 2500,  # típico de quem não informou renda (02/03)
    "MonthlyIncome": None,  # branco
    "NumberOfOpenCreditLinesAndLoans": 4,
    "NumberOfTimes90DaysLate": 0,
    "NumberRealEstateLoansOrLines": 0,
    "NumberOfTime60-89DaysPastDueNotWorse": 0,
    "NumberOfDependents": None,
}

print("=== fichas de mentira ===")
for nome, ficha in [("Ana (perfil seguro)", ana), ("Bruno (02 em pessoa)", bruno), ("Carla (renda em branco)", carla)]:
    d = decidir(ficha)
    print(f"{nome:28s}  nota {d['nota_de_risco']:.3f}  ≥ {d['corte']:.2f}?  → {d['decisao']}")

print()
print("bandeirinhas da Carla (têm que ser 1, 1, 0):")
print({k: decidir(carla)["ficha_limpa"][k] for k in ("income_was_missing", "dependents_was_missing", "util_gt_10", "MonthlyIncome", "DebtRatio")})

# Uma ficha real da prova — só para ver o motor no mundo que ele já conhece.
X_prova = pd.read_csv(PROCESSED / "X_test.csv")
y_prova = pd.read_csv(PROCESSED / "y_test.csv").squeeze("columns")
idx = 7
nota_real = float(motor.predict_proba(X_prova.iloc[[idx]])[0, 1])
print()
print("=== ficha real da prova, linha", idx, "===")
print("nota", round(nota_real, 3), "decisão", "RECUSAR" if nota_real >= politica["corte"] else "LIBERAR")
print("o que aconteceu em 2 anos (só existe porque é prova histórica):", int(y_prova.iloc[idx]))


=== fichas de mentira ===
Ana (perfil seguro)           nota 0.073  ≥ 0.43?  → LIBERAR
Bruno (02 em pessoa)          nota 0.958  ≥ 0.43?  → RECUSAR
Carla (renda em branco)       nota 0.148  ≥ 0.43?  → LIBERAR

bandeirinhas da Carla (têm que ser 1, 1, 0):
{'income_was_missing': 1.0, 'dependents_was_missing': 1.0, 'util_gt_10': 0.0, 'MonthlyIncome': 5400.0, 'DebtRatio': 2500.0}

=== ficha real da prova, linha 7 ===
nota 0.802 decisão RECUSAR
o que aconteceu em 2 anos (só existe porque é prova histórica): 1


### Como ler as três decisões

Nesta execução (se o 04/05 não mudaram):

- **Ana** deve sair **LIBERAR**, nota bem abaixo de 0,43 — idosa, cartão quieto, zero atraso. O 02 já tinha taxa ~2% nesse perfil.
- **Bruno** deve sair **RECUSAR**, nota alta — é a tabela 6b andando: jovem + cartão + 90+.
- **Carla** costuma **LIBERAR** ou ficar perto da linha: renda em branco no 02 caloteava *menos* que a média. A dívida 2500 leva teto ~8138 e a bandeirinha fica 1. Se a bandeirinha viesse 0, a função preencheu na ordem errada.

A ficha da prova: a decisão é a política. O 0/1 do gabarito é **aula**, não produto. Em produção esse 0/1 só aparece daqui a dois anos.


---
# 4. Explicar um “não” (sem SHAP)


O boosting é o motor. A **logística com peso** é a boca. Para o auditor (e para o 01: “modelo explicável”), olhamos quais colunas *empurraram* a nota da logística para cima nesta ficha.

Lembre do 04: depois de igualar unidades, um peso positivo = “isso aumenta risco”. Somamos (valor igualado × peso) e listamos as três maiores empurradas.


In [4]:
def explicar(ficha: dict, n=3) -> pd.DataFrame:
    """Quais colunas mais empurraram a logística para o 'não'."""
    X = preparar_ficha(ficha, regua, colunas)
    pipe = explicavel
    scaler = pipe.named_steps["scaler"]
    clf = pipe.named_steps["clf"]
    iguais = scaler.transform(X)[0]  # mesma régua de unidades do treino
    empurras = pd.Series(iguais * clf.coef_.ravel(), index=colunas, name="empurra")
    tab = pd.DataFrame(
        {
            "valor_na_ficha": X.iloc[0].values,
            "peso": clf.coef_.ravel(),
            "empurra": empurras.values,
        },
        index=colunas,
    ).sort_values("empurra", ascending=False)
    return tab.head(n)


print("Bruno — por que a logística puxa risco para cima:")
print(explicar(bruno).round(3).to_string())
print()
print("Ana — as mesmas colunas, do outro lado:")
print(explicar(ana).round(3).to_string())


Bruno — por que a logística puxa risco para cima:
                                      valor_na_ficha   peso  empurra
RevolvingUtilizationOfUnsecuredLines           1.096  0.749    1.651
NumberOfTime30-59DaysPastDueNotWorse           2.000  0.394    0.991
NumberOfTimes90DaysLate                        1.000  0.485    0.915

Ana — as mesmas colunas, do outro lado:
                        valor_na_ficha   peso  empurra
DebtRatio                         0.22 -0.118    0.039
util_gt_10                        0.00 -0.047    0.002
dependents_was_missing            0.00 -0.008    0.001


### Como ler a explicação

No Bruno, **cartão** e **atraso 90+** (e em geral os atrasos) devem aparecer no topo — o 02 e o 04 já tinham combinado isso. Na Ana, os “empurrões” são pequenos ou negativos (idade, renda).

**Frase para o cliente** (o 01 pediu tom de negócio, não de equação):

> “Sua proposta ficou acima da nossa linha de risco. O que mais pesou foi o uso alto do limite do cartão e um atraso de 90 dias ou mais no histórico. Isso, neste cadastro, se parece com fichas que no passado deram calote grave em 2 anos.”

**O que não dizer:** “Você tem 71% de chance de calote.” As notas do boosting **não** são porcentagem calibrada (caderno 05). A linha 0,43 não quer dizer “43%”.


---
# 5. Lote do dia (o “deploy” que este projeto realmente precisa)


Uma pasta `data/novas/` com um CSV cru do dia. A função abaixo devolve um CSV de decisões. É isso que um banco pequeno rodaria às 2h da manhã.


In [5]:
def decidir_lote(planilha_crua: pd.DataFrame) -> pd.DataFrame:
    """Cada linha do CSV do dia → nota + decisão. Mesma receita, uma a uma."""
    saidas = []
    for i, rec in planilha_crua.iterrows():
        d = decidir(rec.to_dict())
        saidas.append(
            {
                "linha": i,
                "nota_de_risco": d["nota_de_risco"],
                "corte": d["corte"],
                "decisao": d["decisao"],
            }
        )
    return pd.DataFrame(saidas)


# Simulamos o "CSV do dia" com 8 fichas da prova (já limpas).
# Em produção o arquivo viria CRU e passaria por preparar_ficha.
amostra = X_prova.head(8).copy()
lote = pd.DataFrame(
    {
        "nota_de_risco": motor.predict_proba(amostra)[:, 1],
    }
)
lote["decisao"] = np.where(lote["nota_de_risco"] >= politica["corte"], "RECUSAR", "LIBERAR")
lote["gabarito_2_anos"] = y_prova.head(8).to_numpy()  # NÃO existiria no dia D
print(lote.round(3).to_string())
print()
print("recusas no pedacinho:", int((lote["decisao"] == "RECUSAR").sum()), "de", len(lote))


   nota_de_risco  decisao  gabarito_2_anos
0          0.140  LIBERAR                0
1          0.092  LIBERAR                0
2          0.269  LIBERAR                0
3          0.143  LIBERAR                0
4          0.289  LIBERAR                0
5          0.508  RECUSAR                0
6          0.129  LIBERAR                0
7          0.802  RECUSAR                1

recusas no pedacinho: 2 de 8


### Como ler o lote

A coluna `gabarito_2_anos` é trapaça pedagógica. No dia em que o lote roda, ela **não existe**. Se você usar o gabarito para “passar o corte um pouquinho”, voltou o vazamento do 04 — agora em produção.

O que você *pode* contar no dia D: quantas recusas, nota média, % com bandeirinha de renda em branco. Isso é monitoramento de **entrada**, não de desempenho.


---
# 6. O que vigiar depois que a caixa está no ar


## 6.1 Três relógios diferentes

| Relógio | O que muda | Quando você percebe | O que fazer |
|---|---|---|---|
| **Entrada** (data drift) | As fichas *novas* não parecem mais com o treino (mais jovens, mais renda em branco, cartão mais estourado) | **Já no mês 1** — basta comparar colunas | Investigar a origem. Ainda *não* é “o aluno piorou”. |
| **Conceito** (concept drift) | O *mundo* mudou: a mesma ficha agora caloteia mais (crise, pandemia, novo produto) | Só quando o calote **acontece** | A fila e o KS caem. Aí sim retreinar. |
| **Desempenho** | Fila, KS, apontar, custo 20×1 | Neste problema: **até 2 anos** (o alvo do 01 é atraso grave em 24 meses) | Não julgue o aluno em 30 dias pelo calote de 2 anos. |

Essa demora de 2 anos é a lição mais importante do 06. O 01 escolheu `SeriousDlqin2yrs`. Você não pode, em janeiro, dizer “o Recall caiu” sobre propostas de dezembro. Ainda não deu tempo.

O que dá para olhar cedo: atraso de 30 dias, recusas demais, % de renda em branco, nota média. São *sinais*, não o critério do 01.


## 6.2 Um alarme simples: as fichas novas ainda parecem o treino?


Comparamos a **média** de algumas colunas no treino (a turma que o aluno estudou) com um “mês novo” (aqui: a prova, que *deveria* ser parecida — o 03 usou `stratify`).

Se a prova já divergisse feio, o split do 03 teria falhado. Em produção o “mês novo” é o CSV de março, abril…


In [6]:
X_treino = pd.read_csv(PROCESSED / "X_train.csv")
vigia = [
    "RevolvingUtilizationOfUnsecuredLines",
    "age",
    "NumberOfTimes90DaysLate",
    "MonthlyIncome",
    "income_was_missing",
    "util_gt_10",
]
cmp = pd.DataFrame(
    {
        "media_treino": X_treino[vigia].mean(),
        "media_prova": X_prova[vigia].mean(),
    }
)
cmp["diferenca"] = cmp["media_prova"] - cmp["media_treino"]
print(cmp.round(4).to_string())
print()
print("bandeirinha renda em branco no treino/prova:",
      float(X_treino["income_was_missing"].mean()),
      float(X_prova["income_was_missing"].mean()))


                                      media_treino  media_prova  diferenca
RevolvingUtilizationOfUnsecuredLines        0.3201       0.3221     0.0019
age                                        52.3150      52.2176    -0.0974
NumberOfTimes90DaysLate                     0.0900       0.0914     0.0014
MonthlyIncome                            6144.7935    6133.9717   -10.8218
income_was_missing                          0.1979       0.1994     0.0015
util_gt_10                                  0.0016       0.0015    -0.0001

bandeirinha renda em branco no treino/prova: 0.19790164918040984 0.19943333333333332


### Como ler o alarme

As médias devem estar **coladas** (o 03 fez o dever). Em produção, um salto de `income_was_missing` de 20% para 40% quer dizer: o cadastro mudou, ou o parceiro parou de pedir renda — não necessariamente que o boosting “esqueceu” de prever.

**Regra prática de retreino (aprendizado, não dogma):**

1. Todo mês: alarme de entrada (esta tabelinha + % de recusas).
2. Todo trimestre: se existir um *proxy* curto (atraso 30 dias), olhar a fila nele — com desconfiança.
3. Quando o alvo de 2 anos amadurecer: recálculo honesto de fila, KS, custo. Só então mexer na linha ou no aluno.
4. Se mexer: **nova caixa versionada** (régua + aluno + linha). Não sobrescreva a caixa velha: o auditor pergunta “e as recusas de março?”.


---
# 7. Documentação, lei e o que a gente *não* mediu


## 7.1 O que outra pessoa precisa para repetir o projeto

- Os cinco cadernos, nesta ordem, com kernel `.venv`
- `data/processed/` + `impute_params.json`
- `models/*.joblib` + `metrics_fase4.json` + `politica_fase5.json`
- A frase: “linha 0,43 nascida no **treino**, custo 20/1, motor boosting, boca logística”
- O que **não** fazer: dropar 20% sem renda; SMOTE no arquivo inteiro; escolher a linha na prova; scaler fora da receita

Isso *é* o manual. Não precisa de um wiki se os cadernos continuarem legíveis.

## 7.2 Explicar a recusa (01 + regulação)

Em vários países recusar crédito sem motivo é problema. Aqui o motivo cabe numa frase (seção 4). SHAP seria o zoom numa ficha; os **pesos** já bastam para a aula e para boa parte da auditoria.

Idade empurra a nota para *baixo* (idoso mais seguro, 02). Isso é estatística, não autorização para discriminar. O 01 não pediu fairness — o 06 só **marca a dívida**: se for produção de verdade, corte a análise por faixa etária, e não use idade se a lei local proibir.

## 7.3 O que este projeto deixou de propósito

- Calibrar a nota para ela *ser* uma PD (“0,20 = 20%”)
- Fairness, lift chart, XGBoost, API
- Custo 20×1 real (é régua de aprendizado; o banco tem a planilha dele)
- Monitorar calote em 2 anos — o dataset é um recorte estático do Kaggle


---
# 8. O que eu (você) deveria saber dizer de cabeça


## 8.1 Respostas às perguntas-guia deste caderno

**Como o modelo entra no fluxo real?**  
Como **lote do dia** (seção 5). API só se a decisão for no clique. O miolo é `decidir()`.

**Como versionar?**  
Uma caixa: aluno + régua + linha + data. Trocar uma peça sem as outras é bug.

**Como monitorar?**  
Entrada já no mês 1. Desempenho no alvo do 01 só quando os 2 anos amadurecerem. Não confunda os relógios.

**O que documentar?**  
Esta mesa 01→05, os três arquivos JSON, e as proibições de leakage.

## 8.2 O checklist do 01, fechado

| Item do 01 | Caderno que fechou | Número |
|---|---|---|
| Fila > 0,80 | 04 mediu, 05 desenhou | 0,869 |
| KS > 30% | 05 | 58% |
| Apontar ≥ 75% | 05, linha 0,43 | 82,7% |
| Explicável | 04 pesos + 06 frase ao cliente | cartão e 90+ |
| Entender o CRISP-DM | 01…06 | você está aqui |

## 8.3 O que faríamos diferente da próxima vez

1. Um **terceiro pedaço** (validação) só para a linha — e a prova intocada até o relatório. Com 150 mil linhas dava. O 05 foi honesto (linha no treino), mas um pedaço do meio seria ainda mais limpo.
2. Combinar o custo 20×1 com o **gerente**, não com o chute didático.
3. Anotar idade / fairness *antes* de colocar `age` no motor, se o destino for banco de verdade.
4. Calibrar a nota se o produto for “limite automático por PD”.


## Se você souber explicar isto, o ciclo fechou

1. **Por que o `.joblib` sozinho não é implantação?**  
   Porque a ficha crua não é o que o aluno comeu. Sem régua e sem linha, você inventa outra política.

2. **Por que a mediana 5400 continua em 2027?**  
   Porque é a régua do **treino desta caixa**. Se o salário mediano mudar, você **abre uma caixa nova** (retreino), não altera 5400 no meio do mês.

3. **Por que não dá para saber o Recall em 30 dias?**  
   O alvo do 01 é calote em **2 anos**. 30 dias medem outra coisa.

4. **Por que Ana libera e Bruno recusa sem “mágica”?**  
   Porque o 02 já tinha medido essas taxas. O 06 só aplicou a linha do 05 numa ficha.

5. **Por que não dizer “43% de chance” ao cliente?**  
   A linha é um poste. A nota do boosting, com peso extra, **não** está calibrada (05).

6. **O projeto “acabou”?**  
   O aprendizado CRISP-DM desta turma, sim. O modelo no mundo, não: caixa versionada, alarme de entrada, retreino quando o alvo amadurecer.


In [7]:
recado = {
    "fase": 6,
    "caixa": {
        "motor": "models/hist_gb.joblib",
        "explicavel": "models/logreg_balanced.joblib",
        "regua": "data/processed/impute_params.json",
        "politica": "models/politica_fase5.json",
        "linha": politica["corte"],
    },
    "como_entra_no_banco": "lote do dia (decidir / decidir_lote); API só se a decisão for no clique",
    "o_que_vigiar_cedo": "média das colunas, % renda em branco, % recusas, nota média",
    "o_que_so_depois_de_2_anos": "fila, KS, apontar caloteiro, custo 20x1",
    "nao_fazer": [
        "recalcular mediana/P99 em cada ficha nova",
        "usar corte 0,5",
        "dizer que a nota é PD calibrada",
        "julgar Recall em 30 dias",
        "sobrescrever a caixa velha sem versão",
    ],
}
(MODELS / "recado_fase6.json").write_text(json.dumps(recado, ensure_ascii=False, indent=2))
print(json.dumps(recado, ensure_ascii=False, indent=2))
print()
print("CRISP-DM desta turma: 01 entender → 02 olhar → 03 limpar → 04 ensinar → 05 decidir → 06 entregar.")


{
  "fase": 6,
  "caixa": {
    "motor": "models/hist_gb.joblib",
    "explicavel": "models/logreg_balanced.joblib",
    "regua": "data/processed/impute_params.json",
    "politica": "models/politica_fase5.json",
    "linha": 0.43
  },
  "como_entra_no_banco": "lote do dia (decidir / decidir_lote); API só se a decisão for no clique",
  "o_que_vigiar_cedo": "média das colunas, % renda em branco, % recusas, nota média",
  "o_que_so_depois_de_2_anos": "fila, KS, apontar caloteiro, custo 20x1",
  "nao_fazer": [
    "recalcular mediana/P99 em cada ficha nova",
    "usar corte 0,5",
    "dizer que a nota é PD calibrada",
    "julgar Recall em 30 dias",
    "sobrescrever a caixa velha sem versão"
  ]
}

CRISP-DM desta turma: 01 entender → 02 olhar → 03 limpar → 04 ensinar → 05 decidir → 06 entregar.
